# Word-Level Text Generation using RNNs

This lab demonstrates word-level text generation using LSTM networks.

**Dataset**: Alice's Adventures in Wonderland by Lewis Carroll (from Project Gutenberg)

In [1]:
# 设置代理
import os
os.environ['HTTP_PROXY'] = 'http://127.0.0.1:7890'
os.environ['HTTPS_PROXY'] = 'http://127.0.0.1:7890'
print("代理已设置")

代理已设置


In [2]:
import torch
import torchtext
import collections
import requests
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## Load the Dataset

We'll use Alice's Adventures in Wonderland from Project Gutenberg.

In [3]:
def load_book(url=None, local_path=None):
    """Load text from URL or local file"""
    if local_path and os.path.exists(local_path):
        with open(local_path, 'r', encoding='utf-8') as f:
            text = f.read()
    elif url:
        print(f"Downloading from {url}...")
        response = requests.get(url)
        text = response.text
    else:
        raise ValueError("Either url or local_path must be provided")
    
    # Extract main text (remove Gutenberg header/footer)
    start_marker = "*** START OF"
    end_marker = "*** END OF"
    
    start_idx = text.find(start_marker)
    if start_idx != -1:
        start_idx = text.find('\n', start_idx) + 1
    else:
        start_idx = 0
    
    end_idx = text.find(end_marker)
    if end_idx == -1:
        end_idx = len(text)
    
    return text[start_idx:end_idx].strip()

# Try local file first, then download
local_file = './data/alice.txt'
url = 'https://www.gutenberg.org/files/11/11-0.txt'

os.makedirs('./data', exist_ok=True)
text = load_book(url=url, local_path=local_file)

# Save for future use
with open(local_file, 'w', encoding='utf-8') as f:
    f.write(text)

print(f"Loaded {len(text)} characters")
print(f"First 500 characters:\n{text[:500]}...")

Loaded 144599 characters
First 500 characters:
[Illustration]




Alice’s Adventures in Wonderland

by Lewis Carroll

THE MILLENNIUM FULCRUM EDITION 3.0

Contents

 CHAPTER I.     Down the Rabbit-Hole
 CHAPTER II.    The Pool of Tears
 CHAPTER III.   A Caucus-Race and a Long Tale
 CHAPTER IV.    The Rabbit Sends in a Little Bill
 CHAPTER V.     Advice from a Caterpillar
 CHAPTER VI.    Pig and Pepper
 CHAPTER VII.   A Mad Tea-Party
 CHAPTER VIII.  The Queen’s Croquet-Ground
 CHAPTER IX.    The Mock Turtle’s Story
 CHAPTER X.     The Lobster ...


## Build Word Vocabulary

In [4]:
tokenizer = torchtext.data.utils.get_tokenizer('basic_english')

# Tokenize text
words = tokenizer(text)
print(f"Total words: {len(words)}")
print(f"First 20 words: {words[:20]}")

# Build vocabulary
counter = collections.Counter(words)
vocab = torchtext.vocab.Vocab(counter, min_freq=2)

vocab_size = len(vocab)
print(f"\nVocabulary size: {vocab_size}")
print(f"Encoding of 'alice': {vocab.stoi.get('alice', 'UNK')}")
print(f"Word with index 10: {vocab.itos[10]}")

Total words: 31821
First 20 words: ['[illustration]', 'alice’s', 'adventures', 'in', 'wonderland', 'by', 'lewis', 'carroll', 'the', 'millennium', 'fulcrum', 'edition', '3', '.', '0', 'contents', 'chapter', 'i', '.', 'down']

Vocabulary size: 1619
Encoding of 'alice': 14
Word with index 10: of


## Prepare Training Data

For word-level generation, we create input-output pairs where:
- Input: sequence of `seq_len` words
- Output: next word after each input sequence

In [5]:
seq_len = 20  # Number of words in each input sequence

# Convert words to indices
word_indices = [vocab.stoi.get(w, vocab.stoi['<unk>']) for w in words]

def get_batches(data, seq_len, batch_size=32):
    """Generate batches of sequences"""
    n_batches = (len(data) - 1) // (seq_len * batch_size)
    
    # Truncate data to fit batches
    data = data[:n_batches * batch_size * seq_len + 1]
    
    # Create input and target sequences
    inputs = []
    targets = []
    
    for i in range(0, len(data) - seq_len, seq_len):
        inputs.append(data[i:i + seq_len])
        targets.append(data[i + 1:i + seq_len + 1])
    
    return torch.tensor(inputs, dtype=torch.long), torch.tensor(targets, dtype=torch.long)

# Split into train and validation
split_idx = int(len(word_indices) * 0.9)
train_data = word_indices[:split_idx]
val_data = word_indices[split_idx:]

X_train, y_train = get_batches(train_data, seq_len)
X_val, y_val = get_batches(val_data, seq_len)

print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")
print(f"\nExample input shape: {X_train[0].shape}")
print(f"Example input: {X_train[0][:10]}...")

Training samples: 1408
Validation samples: 128

Example input shape: torch.Size([20])
Example input: tensor([   0,  320,  562,   15, 1123,   75,    0,    0,    3,    0])...


In [6]:
from torch.utils.data import TensorDataset, DataLoader

batch_size = 64

train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"Number of training batches: {len(train_loader)}")

Number of training batches: 22


## Define the LSTM Generator Model

In [7]:
class WordLSTMGenerator(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers=2, dropout=0.3):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.lstm = torch.nn.LSTM(
            embed_dim, 
            hidden_dim, 
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0,
            batch_first=True
        )
        self.fc = torch.nn.Linear(hidden_dim, vocab_size)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x, hidden=None):
        embedded = self.dropout(self.embedding(x))
        output, hidden = self.lstm(embedded, hidden)
        output = self.dropout(output)
        logits = self.fc(output)
        return logits, hidden

model = WordLSTMGenerator(vocab_size, embed_dim=128, hidden_dim=256, num_layers=2).to(device)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

WordLSTMGenerator(
  (embedding): Embedding(1619, 128)
  (lstm): LSTM(128, 256, num_layers=2, batch_first=True, dropout=0.3)
  (fc): Linear(in_features=256, out_features=1619, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
)

Total parameters: 1,544,915


## Training Loop

In [8]:
def train_epoch(model, train_loader, optimizer, criterion):
    model.train()
    total_loss = 0
    
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        
        optimizer.zero_grad()
        logits, _ = model(inputs)
        
        # Reshape for cross entropy: (batch * seq_len, vocab_size)
        loss = criterion(logits.view(-1, vocab_size), targets.view(-1))
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(train_loader)

def evaluate(model, val_loader, criterion):
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            logits, _ = model(inputs)
            loss = criterion(logits.view(-1, vocab_size), targets.view(-1))
            total_loss += loss.item()
    
    return total_loss / len(val_loader)

## Text Generation Function

We implement both hard (argmax) and soft (temperature-based) generation.

In [9]:
def generate_text(model, start_text, num_words=50, temperature=None):
    """Generate text from a starting phrase
    
    Args:
        model: The trained model
        start_text: Starting phrase
        num_words: Number of words to generate
        temperature: If None, use argmax (hard). If float, use soft sampling.
    """
    model.eval()
    
    # Tokenize start text
    words = tokenizer(start_text)
    indices = [vocab.stoi.get(w, vocab.stoi['<unk>']) for w in words]
    
    hidden = None
    generated = list(words)
    
    with torch.no_grad():
        # Process initial sequence
        x = torch.tensor([indices], dtype=torch.long).to(device)
        logits, hidden = model(x, hidden)
        
        # Generate words one by one
        for _ in range(num_words):
            # Get logits for last position
            last_logits = logits[0, -1, :]
            
            if temperature is None:
                # Hard generation: take argmax
                next_idx = torch.argmax(last_logits).item()
            else:
                # Soft generation: sample with temperature
                probs = torch.softmax(last_logits / temperature, dim=0)
                next_idx = torch.multinomial(probs, 1).item()
            
            next_word = vocab.itos[next_idx]
            generated.append(next_word)
            
            # Feed back as input
            x = torch.tensor([[next_idx]], dtype=torch.long).to(device)
            logits, hidden = model(x, hidden)
    
    return ' '.join(generated)

In [10]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = torch.nn.CrossEntropyLoss()

epochs = 10
best_val_loss = float('inf')

for epoch in range(epochs):
    train_loss = train_epoch(model, train_loader, optimizer, criterion)
    val_loss = evaluate(model, val_loader, criterion)
    
    print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f} - Val Loss: {val_loss:.4f}")
    
    # Generate sample text
    if (epoch + 1) % 2 == 0:
        sample = generate_text(model, "alice was", 50)
        print(f"Sample: {sample}\n")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), './data/best_model.pt')

Epoch 1/10 - Train Loss: 6.4918 - Val Loss: 5.7079
Epoch 2/10 - Train Loss: 5.7021 - Val Loss: 5.6414
Sample: alice was , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , , ,

Epoch 3/10 - Train Loss: 5.6365 - Val Loss: 5.6203
Epoch 4/10 - Train Loss: 5.5880 - Val Loss: 5.5635
Sample: alice was , and , and , and , and , and , and , and , and , and , and , and , and , and , and , and , and , and , and , and , and , and , and , and , and , and

Epoch 5/10 - Train Loss: 5.5149 - Val Loss: 5.4566
Epoch 6/10 - Train Loss: 5.4225 - Val Loss: 5.3469
Sample: alice was , ” said the the the <unk> , ” said the the the <unk> , ” said the the the <unk> , ” said the the the <unk> , ” said the the the <unk> , ” said the the the <unk> , ” said the the the <unk> ,

Epoch 7/10 - Train Loss: 5.3384 - Val Loss: 5.2840
Epoch 8/10 - Train Loss: 5.2646 - Val Loss: 5.2057
Sample: alice was , ” said the <unk> , ” said the <unk> , ” said the <unk> , ” said the <unk> 

In [11]:
# Load best model
model.load_state_dict(torch.load('./data/best_model.pt'))

print("=" * 60)
print("HARD GENERATION (Argmax)")
print("=" * 60)
print(generate_text(model, "alice was beginning to", num_words=50, temperature=None))

print("\n" + "=" * 60)
print("SOFT GENERATION WITH DIFFERENT TEMPERATURES")
print("=" * 60)

for temp in [0.5, 0.8, 1.0, 1.5]:
    print(f"\n--- Temperature = {temp} ---")
    print(generate_text(model, "the queen said", num_words=40, temperature=temp))

HARD GENERATION (Argmax)
alice was beginning to <unk> , ” said the <unk> , ” said the <unk> , ” said the <unk> , ” said the <unk> , ” said the <unk> , ” said the <unk> , ” said the <unk> , ” said the <unk> , ” said the <unk> , ” said the

SOFT GENERATION WITH DIFFERENT TEMPERATURES

--- Temperature = 0.5 ---
the queen said to , ” said the <unk> , ” said alice , and said the <unk> . ” said the <unk> , ” said the queen , ” said the duchess , and <unk> , and <unk> . ” said the

--- Temperature = 0.8 ---
the queen said i in , and was the too <unk> for to one three for herself her <unk> , and majesty . <unk> say that with to <unk> ! ” the “but for voice , and her , ” said the she

--- Temperature = 1.0 ---
the queen said over , were that idea , ” “i along you after he join , is as the so you began again the do , ” the they’ll all at the , into alice voice , <unk> i makes learn flowers

--- Temperature = 1.5 ---
the queen said she lives looking one dreadfully * when ( dear . look . rabbit bo

/tmp/ipykernel_57863/2358372690.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('./data/best_model.pt'))


## Interactive Generation

Try different starting phrases and see what the model generates!

In [12]:
start_phrases = [
    "the white rabbit",
    "down the rabbit hole",
    "the mad hatter",
    "off with her head"
]

print("Generated Texts (Temperature = 0.8):\n")
for phrase in start_phrases:
    print(f"Starting: \"{phrase}\"")
    print(generate_text(model, phrase, num_words=30, temperature=0.8))
    print("-" * 50)

Generated Texts (Temperature = 0.8):

Starting: "the white rabbit"
the white rabbit you ! ” the <unk> , and looked must , ” alice alice down , alice i , i my head <unk> with , and the gave , ” there
--------------------------------------------------
Starting: "down the rabbit hole"
down the rabbit hole , as quite sighed of the other <unk> at her ) question , mouse on the was ! ” i the so , ” she the in a was ,
--------------------------------------------------
Starting: "the mad hatter"
the mad hatter the sir his she seemed <unk> use , “i <unk> , ” said the hare my dry , and dreadful for anxiously suddenly . game in i of have .
--------------------------------------------------
Starting: "off with her head"
off with her head , ” you the dormouse off know , and she see when a a <unk> , ” ) ? ” “i caterpillar she <unk> . least the <unk> , and
--------------------------------------------------


## Comparison: Character-Level vs Word-Level

| Aspect | Character-Level | Word-Level |
|--------|----------------|------------|
| Vocabulary Size | Small (~100) | Large (thousands+) |
| Sequence Length | Long | Shorter |
| Memory | More efficient | Requires more memory |
| Coherence | Can struggle with word formation | Better semantic coherence |
| Training Time | Faster per epoch | Slower per epoch |

### Findings:
1. **Word-level models** produce more coherent text with proper word structure
2. **Temperature** affects creativity - lower = more conservative, higher = more random
3. **LSTM** captures some syntactic patterns from the book
4. **Longer training** and **larger hidden size** would improve quality